# 🤖 Agentic Triage — Walkthrough Consumer Surface

**Repo:** [AdamKrysztopa/dependence-forecastability](https://github.com/AdamKrysztopa/dependence-forecastability)
&nbsp;·&nbsp;
**Paper:** [arXiv:2601.10006](https://arxiv.org/abs/2601.10006)

---

## Introduction

This notebook is the **end-to-end walkthrough consumer** for triage in this repository.
It demonstrates how to call `run_triage()` and inspect deterministic outputs across
representative scenarios, while keeping runtime logic inside `src/forecastability`.

This notebook is intentionally **not** the deep-dive payload/adapter notebook.
For deterministic payload/serializer/interpretation internals, use:
`notebooks/triage/10_agent_ready_triage_interpretation.ipynb`.

### What this notebook covers

| Section | Content |
|---------|---------|
| §1 | **One-cell triage** — `run_triage()` as a single entry point |
| §2 | **Readiness gate** — blocked vs warning vs clear |
| §3 | **Univariate triage** across canonical synthetic structures |
| §4 | **Exogenous triage** on target + driver series |
| §5 | **Event emission & timing** with `CollectingEventEmitter` |
| §6 | **Result visualisation** for triage curves and summaries |
| §7 | **Optional narration consumption** (guarded; no adapter internals) |

## Setup ⚙️

```bash
uv sync --group notebook
uv run python -m ipykernel install --user --name forecastability
```

In [ ]:
%matplotlib inline
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

mpl.rcParams.update(
    {
        "figure.figsize": (12, 5),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": "sans-serif",
        "figure.dpi": 120,
        "axes.prop_cycle": plt.cycler(
            color=[
                "#1f77b4",
                "#ff7f0e",
                "#2ca02c",
                "#d62728",
                "#9467bd",
                "#8c564b",
                "#e377c2",
                "#17becf",
                "#bcbd22",
                "#7f7f7f",
            ]
        ),
    }
)

# ── Resolve project root so relative data paths work from any CWD ────────────
_nb_path = Path(__file__) if "__file__" in dir() else Path.cwd() / "notebooks"
REPO_ROOT = _nb_path.parent
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print(f"Working directory: {Path.cwd()}")
print("✅ Matplotlib configured.")

In [ ]:
# ── Triage imports ────────────────────────────────────────────────────────────
from forecastability.triage import (
    AnalysisGoal,
    MethodPlan,
    ReadinessReport,
    ReadinessStatus,
    TriageRequest,
    TriageResult,
    assess_readiness,
    run_triage,
)
from forecastability.adapters.event_emitter import CollectingEventEmitter
from forecastability.triage.events import TriageStageCompleted, TriageStageStarted

# ── Dataset generators (same as Notebook 01) ─────────────────────────────────
from forecastability.utils.datasets import (
    ar1_theoretical_ami,
    generate_ar1,
    generate_henon_map,
    generate_sine_wave,
    generate_white_noise,
)

print("✅ All imports successful.")

## §1 · One-Cell Triage — the single entry point

`run_triage(TriageRequest(...))` is the **sole entry point** to the entire
pipeline. In one call it:

1. Validates the series for length and lag feasibility
2. Routes to the correct compute backend (univariate / exogenous)
3. Runs the MI estimator
4. Interprets the result

The returned `TriageResult` contains every intermediate artefact.

In [ ]:
# ── Generate AR(1) series (same as Notebook 01) ───────────────────────────────
ar1 = generate_ar1(n_samples=500, phi=0.7, random_state=42)

# ── One-cell triage ───────────────────────────────────────────────────────────
result: TriageResult = run_triage(
    TriageRequest(
        series=ar1,
        goal=AnalysisGoal.univariate,
        max_lag=40,
        n_surrogates=99,
        random_state=42,
    )
)

print(f"Readiness status : {result.readiness.status}")
print(f"Route selected   : {result.method_plan.route if result.method_plan else 'N/A'}")
print(f"Blocked          : {result.blocked}")
print(f"Recommendation   : {result.recommendation}")
if result.interpretation:
    interp = result.interpretation
    print(f"Forecastability  : {interp.forecastability_class}")
    print(f"Directness       : {interp.directness_class}")
    print(f"Modeling regime  : {interp.modeling_regime}")

## §2 · Readiness Gate Demo

The readiness gate checks:
- Minimum series length
- Lag feasibility (max_lag < N / 2)
- Significance feasibility (N large enough for surrogates)

When blocked, `run_triage` returns immediately with `blocked=True` and no
compute is performed.

In [ ]:
# ── Trigger a blocked gate with a very short series ───────────────────────────
short_series = np.random.default_rng(0).standard_normal(10)

blocked_result = run_triage(
    TriageRequest(
        series=short_series,
        goal=AnalysisGoal.univariate,
        max_lag=8,
        n_surrogates=99,
        random_state=42,
    )
)

print(f"Blocked  : {blocked_result.blocked}")
print(f"Status   : {blocked_result.readiness.status}")
for w in blocked_result.readiness.warnings:
    print(f"  [{w.code}] {w.message}")

In [ ]:
# ── Warning (not blocked) — feasible but with caveats ────────────────────────
small_series = generate_ar1(n_samples=85, phi=0.5, random_state=7)

warned_result = run_triage(
    TriageRequest(
        series=small_series,
        goal=AnalysisGoal.univariate,
        max_lag=30,
        n_surrogates=99,
        random_state=42,
    )
)

print(f"Blocked  : {warned_result.blocked}")
print(f"Status   : {warned_result.readiness.status}")
for w in warned_result.readiness.warnings:
    print(f"  [{w.code}] {w.message}")
if not warned_result.blocked:
    print(f"Recommendation : {warned_result.recommendation}")

## §3 · Univariate Triage — All Five Canonical Series

Same series used in Notebook 01, now routed exclusively through `run_triage()`.

In [ ]:
# ── Re-generate the five canonical series (same as Notebook 01) ──────────────
def _logistic_map(n: int = 500, r: float = 3.9, x0: float = 0.501) -> np.ndarray:
    x = np.zeros(n)
    x[0] = x0
    for t in range(1, n):
        x[t] = r * x[t - 1] * (1.0 - x[t - 1])
    return x


SERIES: dict[str, np.ndarray] = {
    "White Noise": generate_white_noise(n_samples=500, random_state=42),
    "AR(1) φ=0.7": generate_ar1(n_samples=500, phi=0.7, random_state=42),
    "Logistic Map (r=3.9)": _logistic_map(),
    "Sine + Noise": generate_sine_wave(n_samples=500, cycles=10, noise_std=0.15),
    "Hénon Map": generate_henon_map(n_samples=500),
}

print(f"Generated {len(SERIES)} canonical series:")
for name, ts in SERIES.items():
    print(f"  {name:28s}  n={len(ts):>4d}  mean={ts.mean():.3f}  std={ts.std():.3f}")

In [ ]:
# ── Triage all canonical series through run_triage() ─────────────────────────
triage_results: dict[str, TriageResult] = {}

for name, ts in SERIES.items():
    tr = run_triage(
        TriageRequest(
            series=ts,
            goal=AnalysisGoal.univariate,
            max_lag=40,
            n_surrogates=99,
            random_state=42,
        )
    )
    triage_results[name] = tr

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for name, tr in triage_results.items():
    row: dict[str, object] = {
        "Series": name,
        "Blocked": tr.blocked,
        "Route": tr.method_plan.route if tr.method_plan else "N/A",
    }
    if tr.interpretation:
        row["Forecastability"] = tr.interpretation.forecastability_class
        row["Directness"] = tr.interpretation.directness_class
        row["Modeling Regime"] = tr.interpretation.modeling_regime
    if tr.analyze_result is not None:
        row["Peak AMI"] = round(float(tr.analyze_result.raw.max()), 4)
    rows.append(row)

pd.DataFrame(rows).set_index("Series")

## §4 · Exogenous / CrossAMI Triage — Real-World Pairs

The same real-world datasets used in Notebook 02 are now triaged via the
`AnalysisGoal.exogenous` path. `run_triage` routes these to
`ForecastabilityAnalyzerExog`, which computes **CrossAMI** (raw) and
**pCrossAMI** (partial, conditioning on target's own AR(1)).

| Pair | Target | Driver | Hypothesis |
|------|--------|--------|------------|
| Bike / Temp | cnt | temp | Temperature drives ridership |
| AAPL / SPY | AAPL | SPY | S&P 500 drives individual stock returns |
| BTC / ETH | BTC | ETH | Ethereum co-moves with Bitcoin |

In [ ]:
# ── Load exogenous datasets (same files as Notebook 02) ───────────────────────
# Resolve repo root regardless of CWD (walkthroughs/ or repo root)
_here = Path.cwd().resolve()
REPO_ROOT = _here
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "pyproject.toml").exists():
        break
    REPO_ROOT = REPO_ROOT.parent

EXOG_DIR = REPO_ROOT / "data" / "exog"
CANONICAL_DIR = REPO_ROOT / "data" / "canonical"

# Bike sharing — hourly cnt (target) vs temp (driver)
bike = pd.read_csv(EXOG_DIR / "bike_sharing_hour.csv")
bike_cnt = bike["cnt"].to_numpy(dtype=float)
bike_temp = bike["temp"].to_numpy(dtype=float)

# AAPL / SPY daily log-returns
spy_df = pd.read_csv(EXOG_DIR / "spy_returns.csv", index_col=0, parse_dates=True)
aapl_df = pd.read_csv(CANONICAL_DIR / "aapl_returns.csv", index_col=0, parse_dates=True)

common_idx_eq = aapl_df.index.intersection(spy_df.index)
aapl_arr = aapl_df.loc[common_idx_eq].to_numpy(dtype=float).ravel()
spy_arr = spy_df.loc[common_idx_eq].to_numpy(dtype=float).ravel()

# BTC / ETH
btc_df = pd.read_csv(CANONICAL_DIR / "bitcoin_returns.csv", index_col=0, parse_dates=True)
ETH_PATH = EXOG_DIR / "eth_returns.csv"
if ETH_PATH.exists():
    eth_df = pd.read_csv(ETH_PATH, index_col=0, parse_dates=True)
    common_idx_crypto = btc_df.index.intersection(eth_df.index)
    btc_arr = btc_df.loc[common_idx_crypto].to_numpy(dtype=float).ravel()
    eth_arr = eth_df.loc[common_idx_crypto].to_numpy(dtype=float).ravel()
    HAS_ETH = True
else:
    HAS_ETH = False
    print("eth_returns.csv not found — BTC/ETH pair will be skipped.")

print(f"Bike  : cnt n={len(bike_cnt):>5d}  temp n={len(bike_temp):>5d}")
print(f"AAPL/SPY : aligned n={len(aapl_arr):>5d}")
if HAS_ETH:
    print(f"BTC/ETH  : aligned n={len(btc_arr):>5d}")

In [ ]:
# ── Exogenous triage via run_triage() ─────────────────────────────────────────
exog_pairs: dict[str, tuple[np.ndarray, np.ndarray]] = {
    "Bike cnt ← temp": (bike_cnt[:2000], bike_temp[:2000]),
    "AAPL ← SPY": (aapl_arr, spy_arr),
}
if HAS_ETH:
    exog_pairs["BTC ← ETH"] = (btc_arr, eth_arr)

exog_triage_results: dict[str, TriageResult] = {}

for pair_name, (target, driver) in exog_pairs.items():
    tr = run_triage(
        TriageRequest(
            series=target,
            exog=driver,
            goal=AnalysisGoal.exogenous,
            max_lag=40,
            n_surrogates=99,
            random_state=42,
        )
    )
    exog_triage_results[pair_name] = tr
    peak_raw = float(tr.analyze_result.raw.max()) if tr.analyze_result is not None else float("nan")
    peak_partial = (
        float(tr.analyze_result.partial.max())
        if (tr.analyze_result is not None and tr.analyze_result.partial is not None)
        else float("nan")
    )
    route = tr.method_plan.route if tr.method_plan else "N/A"
    print(
        f"{pair_name:20s}  route={route:30s}"
        f"  peak_CrossAMI={peak_raw:.4f}  peak_pCrossAMI={peak_partial:.4f}"
    )

## §5 · Event Emission & Timing

`CollectingEventEmitter` accumulates lifecycle events in memory.
Pass it to `run_triage` via `event_emitter=` to observe the 4 pipeline stages:
`readiness`, `routing`, `compute`, `interpretation`.

In [ ]:
# ── Run triage with event collection ──────────────────────────────────────────
emitter = CollectingEventEmitter()

result_with_events = run_triage(
    TriageRequest(
        series=SERIES["Hénon Map"],
        goal=AnalysisGoal.univariate,
        max_lag=40,
        n_surrogates=99,
        random_state=42,
    ),
    event_emitter=emitter,
)

print(f"{'Stage':<20}  {'Event type':<30}  {'Duration ms':>12}  Summary")
print("─" * 95)
for evt in emitter.events:
    if isinstance(evt, TriageStageStarted):
        print(f"  {evt.stage:<18}  {'TriageStageStarted':<30}  {'':>12}")
    elif isinstance(evt, TriageStageCompleted):
        print(
            f"  {evt.stage:<18}  {'TriageStageCompleted':<30}"
            f"  {evt.duration_ms:>12.1f}  {evt.result_summary}"
        )

In [ ]:
# ── Timing breakdown from result.timing ───────────────────────────────────────
if result_with_events.timing:
    timing = result_with_events.timing
    total_ms = sum(timing.values())
    print(f"{'Stage':<20}  {'ms':>10}  {'share':>8}")
    print("─" * 44)
    for stage, ms in timing.items():
        print(f"  {stage:<18}  {ms:>10.1f}  {ms / total_ms * 100:>7.1f}%")
    print("─" * 44)
    print(f"  {'TOTAL':<18}  {total_ms:>10.1f}  {'100.0%':>8}")

## §6 · TriageResult Visualisation

Plot the dependence curves stored inside each `TriageResult.analyze_result`.

### 6a — AMI · pAMI for canonical series (from §3)

In [ ]:
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

fig, axes = plt.subplots(2, 5, figsize=(22, 7))

for col, (name, tr) in enumerate(triage_results.items()):
    ar = tr.analyze_result
    if ar is None:
        continue
    lags_raw = np.arange(1, ar.raw.size + 1)
    c = colors[col]

    axes[0, col].plot(lags_raw, ar.raw, lw=2, color=c)
    axes[0, col].fill_between(lags_raw, 0, ar.raw, alpha=0.13, color=c)
    axes[0, col].set_title(name, fontsize=9, fontweight="bold")
    axes[0, col].set_ylim(bottom=0)

    if ar.partial is not None:
        lags_par = np.arange(1, ar.partial.size + 1)
        axes[1, col].plot(lags_par, ar.partial, lw=2, color=c, alpha=0.7)
        axes[1, col].fill_between(lags_par, 0, ar.partial, alpha=0.2, color=c)
        axes[1, col].set_ylim(bottom=0)

for ax in axes.flat:
    ax.set_xlabel("Lag h", fontsize=8)
    ax.tick_params(labelsize=7)

axes[0, 0].set_ylabel("AMI(h)  [raw]", fontsize=10)
axes[1, 0].set_ylabel("pAMI(h)  [partial]", fontsize=10)

fig.suptitle(
    "Univariate Triage — AMI (top) · pAMI (bottom)  |  via run_triage()",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

### 6b — CrossAMI · pCrossAMI for exogenous pairs (from §4)

In [ ]:
exog_colors = ["#17becf", "#bcbd22", "#8c564b"]
n_pairs = len(exog_triage_results)

fig, axes = plt.subplots(2, n_pairs, figsize=(6 * n_pairs, 7), squeeze=False)

for col, (pair_name, tr) in enumerate(exog_triage_results.items()):
    ar = tr.analyze_result
    if ar is None:
        continue
    c = exog_colors[col % len(exog_colors)]
    lags_raw = np.arange(1, ar.raw.size + 1)

    axes[0, col].plot(lags_raw, ar.raw, lw=2, color=c)
    axes[0, col].fill_between(lags_raw, 0, ar.raw, alpha=0.15, color=c)
    axes[0, col].set_title(pair_name, fontsize=10, fontweight="bold")
    axes[0, col].set_ylim(bottom=0)

    if ar.partial is not None:
        lags_par = np.arange(1, ar.partial.size + 1)
        axes[1, col].plot(lags_par, ar.partial, lw=2, color=c, alpha=0.7)
        axes[1, col].fill_between(lags_par, 0, ar.partial, alpha=0.2, color=c)
        axes[1, col].set_ylim(bottom=0)

for ax in axes.flat:
    ax.set_xlabel("Lag h", fontsize=8)
    ax.tick_params(labelsize=7)

axes[0, 0].set_ylabel("CrossAMI(h)  [raw]", fontsize=10)
axes[1, 0].set_ylabel("pCrossAMI(h)  [partial]", fontsize=10)

fig.suptitle(
    "Exogenous Triage — CrossAMI (top) · pCrossAMI (bottom)  |  via run_triage()",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

### 6c — Consolidated CrossAMI peak comparison

In [ ]:
pair_names = list(exog_triage_results.keys())
peak_cross = [
    float(tr.analyze_result.raw.max()) if tr.analyze_result is not None else 0.0
    for tr in exog_triage_results.values()
]
peak_pcross = [
    float(tr.analyze_result.partial.max())
    if (tr.analyze_result is not None and tr.analyze_result.partial is not None)
    else 0.0
    for tr in exog_triage_results.values()
]

x = np.arange(len(pair_names))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w / 2, peak_cross, w, label="Peak CrossAMI", color="#17becf", alpha=0.85)
ax.bar(x + w / 2, peak_pcross, w, label="Peak pCrossAMI", color="#bcbd22", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(pair_names, fontsize=10)
ax.set_ylabel("Peak MI score")
ax.set_title("CrossAMI vs pCrossAMI — Peak Value per Exogenous Pair", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## §7 · PydanticAI Agent (requires API key)

The `create_triage_agent()` factory wraps the triage pipeline in a
[PydanticAI](https://ai.pydantic.dev/) agent. The agent uses `run_triage` as a
tool and returns a structured `TriageExplanation` — natural-language narrative
plus typed fields.

This cell is **guarded** by an `ImportError` check so the notebook remains fully
functional even without `pydantic-ai` installed or an API key configured.

> **To enable:** create a `.env` file at the repo root with:
> ```
> OPENAI_API_KEY=sk-...
> ```
> or `ANTHROPIC_API_KEY` / `XAI_API_KEY` for other providers.

In [ ]:
try:
    from forecastability.adapters.pydantic_ai_agent import TriageDeps, create_triage_agent
    from forecastability.adapters.settings import InfraSettings

    AGENT_AVAILABLE = True
except ImportError:
    AGENT_AVAILABLE = False

print(f"PydanticAI agent available: {AGENT_AVAILABLE}")

In [ ]:
if AGENT_AVAILABLE:
    settings = InfraSettings()  # loads from .env
    api_key_present = bool(
        settings.openai_api_key
        or settings.anthropic_api_key
        or getattr(settings, "xai_api_key", None)
    )
    print(f"API key configured: {api_key_present}")
    if not api_key_present:
        print(
            "⚠️  No API key found in .env — skipping live agent invocation.\n"
            "   Set OPENAI_API_KEY (or ANTHROPIC_API_KEY) in .env to enable."
        )
else:
    api_key_present = False
    print("Skipping settings check — pydantic-ai not installed.")

In [ ]:
if AGENT_AVAILABLE and api_key_present:
    try:
        agent = create_triage_agent(settings=settings)
        
        deps = TriageDeps(
            settings=settings,
            series=SERIES["AR(1) φ=0.7"],
            exog=None,
            goal=AnalysisGoal.univariate,
            max_lag=40,
            n_surrogates=99,
            random_state=42,
        )
        
        # Jupyter already runs an event loop — use top-level await (IPython 7+)
        agent_result = await agent.run(
            "Assess the forecastability of this AR(1) time series and recommend a modeling strategy.",
            deps=deps,
        )
        expl = agent_result.output
        print(f"Forecastability class : {expl.forecastability_class}")
        print(f"Directness class      : {expl.directness_class}")
        print(f"Modeling regime       : {expl.modeling_regime}")
        print(f"Primary lags          : {expl.primary_lags}")
        print(f"Recommendation        : {expl.recommendation}")
        print(f"\nNarrative:\n  {expl.narrative}")
        if expl.caveats:
            print("\nCaveats:")
            for c in expl.caveats:
                print(f"  - {c}")
        else:
            print("Agent invocation skipped — set up API key and re-run to enable live inference.")
    except ImportError as _e:
        print(f"⚠️  Agent unavailable (pydantic-ai not installed): {_e}")
        print("Install with: uv sync --extra agent")


## Takeaways

### Architecture highlights

| Component | Role |
|-----------|------|
| `run_triage(TriageRequest(...))` | **Single entry point** — hides all routing logic from callers |
| `ReadinessReport` | Short-circuits expensive compute when data is unsuitable |
| `MethodPlan.route` | Selects univariate vs exogenous backend transparently |
| `CollectingEventEmitter` | In-process observability — no framework required |
| `create_triage_agent()` | Optional AI layer — structured `TriageExplanation` output |

### When to use each analysis goal

| Goal | Use when |
|------|----------|
| `AnalysisGoal.univariate` | Self-predictability of a target series (AMI · pAMI) |
| `AnalysisGoal.exogenous` | Cross-series predictability — does a driver improve target forecast? (CrossAMI · pCrossAMI) |

### Links

- 📖 **Paper:** [arXiv:2601.10006 — Dr. Peter Catt](https://arxiv.org/abs/2601.10006)
- 🐙 **Repository:** [github.com/AdamKrysztopa/dependence-forecastability](https://github.com/AdamKrysztopa/dependence-forecastability)
- 📓 **Canonical analysis:** `01_canonical_forecastability.ipynb`
- 📓 **Exogenous analysis:** `02_exogenous_analysis.ipynb`

---
*Notebook 03 — Agentic Triage · Python 3.11 · forecastability package*